## DSCI-552-HW7
**studnet name** : Qian Ling

**student ID** : 7728910687

**github name** : qianling-sketch


In [36]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC, LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score, hamming_loss

RANDOM_STATE = 42
LABELS = ["Family", "Genus", "Species"]

In [37]:
# 1(a) Load dataset
df = pd.read_csv("../Data/Frogs_MFCCs.csv")

# Separate features and labels
X = df.drop(columns=LABELS + ["RecordID"])
y = df[LABELS]

# Randomly split 70% training and 30% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE
)

print("Dataset:", df.shape)
print("Training:", X_train.shape, y_train.shape)
print("Testing:", X_test.shape, y_test.shape)

Dataset: (7195, 26)
Training: (5036, 22) (5036, 3)
Testing: (2159, 22) (2159, 3)


### (b)i

**Exact Match Ratio:** The proportion of samples for which all three labels are predicted correctly. Higher is better.

**Hamming Loss:** The proportion of incorrectly predicted labels across all samples and labels. Lower is better.

**Hamming Score:** The proportion of correctly predicted labels, calculated as 1 - Hamming Loss. Higher is better.

These metrics will be used to evaluate the SVM classifiers in the following sections.

In [38]:
# Evaluation function for multi-class, multi-label classification
def evaluate_model(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Exact match: all three labels must be correct
    exact_match = np.mean(np.all(y_true == y_pred, axis=1))

    # Hamming loss: fraction of incorrect labels
    hamming = np.mean(y_true != y_pred)

    return {
        "Exact Match": exact_match,
        "Hamming Loss": hamming,
        "Hamming Score": 1 - hamming
    }

In [39]:
# b(ii)
# Train one classifier for each label using 10-fold CV
def train_svms(X_train, y_train, model, param_grid):
    models = {}
    best_params = []

    for label in LABELS:
        grid = GridSearchCV(
            model, param_grid, cv=10,
            scoring="accuracy", n_jobs=-1
        )
        grid.fit(X_train, y_train[label])

        models[label] = grid.best_estimator_
        best_params.append({
            "Label": label,
            **grid.best_params_
        })

    return models, pd.DataFrame(best_params)


# Predict all three labels
def predict_svms(models, X_test):
    return pd.DataFrame({
        label: models[label].predict(X_test)
        for label in LABELS
    }, index=X_test.index)

In [40]:
# Gaussian SVM with standardization and One-vs-Rest
rbf_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", OneVsRestClassifier(SVC(kernel="rbf")))
])

# Parameters for 10-fold cross-validation
rbf_params = {
    "svm__estimator__C": [0.1, 1, 10, 100],
    "svm__estimator__gamma": [0.001, 0.01, 0.1, 1]
}

# Train and tune models
rbf_models, rbf_best = train_svms(
    X_train, y_train, rbf_svm, rbf_params
)

# Predict and evaluate
rbf_pred = predict_svms(rbf_models, X_test)
rbf_scores = evaluate_model(y_test, rbf_pred)

print("Best Parameters:")
display(rbf_best)

print("Test Results:")
display(pd.DataFrame([rbf_scores]))

Best Parameters:


,Label,svm__estimator__C,svm__estimator__gamma
0,Family,10,0.1
1,Genus,10,0.1
2,Species,10,0.1


Test Results:


,Exact Match,Hamming Loss,Hamming Score
0,0.986568,0.010036,0.989964


In [41]:
# 1(b)ii Results
rbf_scores = evaluate_model(y_test, rbf_pred)

print("Best Parameters:")
display(rbf_best)

print("Test Results:")
display(pd.DataFrame([rbf_scores]))

Best Parameters:


,Label,svm__estimator__C,svm__estimator__gamma
0,Family,10,0.1
1,Genus,10,0.1
2,Species,10,0.1


Test Results:


,Exact Match,Hamming Loss,Hamming Score
0,0.986568,0.010036,0.989964


The best parameters selected by 10-fold cross-validation were C = 10 and gamma = 0.1 for all three labels.

The Gaussian SVM achieved an Exact Match Ratio of 98.66% and a Hamming Loss of 1.00% on the test set, indicating strong classification performance.

In [48]:
# 1(b)iii L1-penalized SVM
l1_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", OneVsRestClassifier(
        LinearSVC(penalty="l1", dual=False, max_iter=20000)
    ))
])

# Tune C using 10-fold cross-validation
l1_params = {
    "svm__estimator__C": [0.001, 0.01, 0.1, 1, 10, 100]
}

# Train and tune models
l1_models, l1_best = train_svms(
    X_train, y_train, l1_svm, l1_params
)

# Predict and evaluate
l1_pred = predict_svms(l1_models, X_test)
l1_scores = evaluate_model(y_test, l1_pred)

print("Best Parameters:")
display(l1_best)

print("Test Results:")
display(pd.DataFrame([l1_scores]))

Best Parameters:


,Label,svm__estimator__C
0,Family,1
1,Genus,10
2,Species,10


Test Results:


,Exact Match,Hamming Loss,Hamming Score
0,0.912923,0.056816,0.943184


The optimal C values selected by 10-fold cross-validation were 1, 10, and 10 for Family, Genus, and Species, respectively.

The L1-penalized SVM achieved an Exact Match Ratio of 91.29% and a Hamming Loss of 5.68%. Compared with the Gaussian SVM, the L1 SVM performed worse, suggesting that the nonlinear RBF kernel is more effective for this dataset.

In [42]:
# 1(b)iv L1-penalized SVM with SMOTE
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.base import clone

# Use SMOTE when possible; otherwise use random oversampling
def make_sampler(y):
    min_count = pd.Series(y).value_counts().min()
    if min_count >= 2:
        return SMOTE(
            random_state=RANDOM_STATE,
            k_neighbors=min(5, min_count - 1)
        )
    return RandomOverSampler(random_state=RANDOM_STATE)


# Train one SMOTE model for each label
smote_models = {}
smote_best = []

for label in LABELS:
    sampler = make_sampler(y_train[label])

    model = ImbPipeline([
        ("scaler", StandardScaler()),
        ("sampler", sampler),
        ("svm", OneVsRestClassifier(
            LinearSVC(penalty="l1", dual=False, max_iter=20000)
        ))
    ])

    grid = GridSearchCV(
        model,
        {"svm__estimator__C": [0.001, 0.01, 0.1, 1, 10, 100]},
        cv=10,
        scoring="accuracy",
        n_jobs=-1
    )

    grid.fit(X_train, y_train[label])

    smote_models[label] = grid.best_estimator_
    smote_best.append({
        "Label": label,
        "Best C": grid.best_params_["svm__estimator__C"]
    })

# Predict and evaluate
smote_pred = predict_svms(smote_models, X_test)
smote_scores = evaluate_model(y_test, smote_pred)

print("Best Parameters:")
display(pd.DataFrame(smote_best))

print("Test Results:")
display(pd.DataFrame([smote_scores]))

Best Parameters:


,Label,Best C
0,Family,1
1,Genus,10
2,Species,1


Test Results:


,Exact Match,Hamming Loss,Hamming Score
0,0.851783,0.077505,0.922495


In [43]:
# Compare three SVM methods
comparison = pd.DataFrame({
    "Gaussian SVM": rbf_scores,
    "L1 SVM": l1_scores,
    "L1 SVM + SMOTE": smote_scores
}).T

display(comparison.round(4))

,Exact Match,Hamming Loss,Hamming Score
Gaussian SVM,0.9866,0.0100,0.9900
L1 SVM,0.9129,0.0568,0.9432
L1 SVM + SMOTE,0.8518,0.0775,0.9225


After applying SMOTE, the optimal C values were 1, 10, and 1 for Family, Genus, and Species, respectively. The model achieved an Exact Match Ratio of 85.18% and a Hamming Loss of 7.77%.

Compared with the original L1 SVM, SMOTE reduced the overall classification performance. This suggests that oversampling did not improve the test results for this dataset, possibly because it introduced additional synthetic samples that made classification more difficult.

Among the three methods, Gaussian SVM performed best, achieving the highest Exact Match Ratio and the lowest Hamming Loss.

In [45]:
# 2(a)-(c) K-Means Clustering
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy.stats import mode

# Find the best k using silhouette score
def find_best_k(X, seed):
    best_k, best_score = 2, -1

    for k in range(2, 51):
        km = KMeans(n_clusters=k, random_state=seed, n_init=10)
        clusters = km.fit_predict(X)

        score = silhouette_score(
            X, clusters, sample_size=1000, random_state=seed
        )

        if score > best_score:
            best_k, best_score = k, score

    return best_k


# Assign majority labels and evaluate clustering
def evaluate_clusters(X, y, k, seed):
    km = KMeans(n_clusters=k, random_state=seed, n_init=10)
    clusters = km.fit_predict(X)

    y_pred = pd.DataFrame(index=y.index, columns=LABELS)

    for cluster in np.unique(clusters):
        mask = clusters == cluster

        for label in LABELS:
            majority = y.loc[mask, label].mode()[0]
            y_pred.loc[mask, label] = majority

    # Hamming distance per sample = number of incorrect labels
    distance = np.mean(np.sum(
        y.to_numpy() != y_pred.to_numpy(), axis=1
    ))

    loss = np.mean(y.to_numpy() != y_pred.to_numpy())

    return {
        "Best k": k,
        "Hamming Distance": distance,
        "Hamming Score": 1 - loss,
        "Hamming Loss": loss
    }

In [46]:
# Monte Carlo simulation: 50 runs
results = []

for i in range(50):
    best_k = find_best_k(X, RANDOM_STATE + i)

    result = evaluate_clusters(
        X, y, best_k, RANDOM_STATE + i
    )

    results.append(result)

mc_results = pd.DataFrame(results)

# Mean and standard deviation
summary = mc_results.agg(["mean", "std"])

print("Monte Carlo Results (50 runs):")
display(summary.round(4))

Monte Carlo Results (50 runs):


,Best k,Hamming Distance,Hamming Score,Hamming Loss
mean,4.0200,0.6747,0.7751,0.2249
std,0.2466,0.0550,0.0183,0.0183


In [47]:
print(len(mc_results))
print(mc_results["Best k"].value_counts().sort_index())

50
Best k
3     1
4    47
5     2
Name: count, dtype: int64


### 2. K-Means Clustering and Monte Carlo Simulation

K-Means was applied to the entire dataset, with the optimal k selected using Silhouette Scores estimated from 1,000 samples. Majority voting was used to assign Family, Genus, and Species labels to each cluster.

Over 50 runs, k = 4 was selected 47 times, with an average k of 4.02. The average Hamming Distance was 0.6747, Hamming Score was 0.7751, and Hamming Loss was 0.2249.

These results suggest that K-Means captures some patterns in the data, but its classification performance is lower than that of the supervised SVM models.

## 3. ISLR 12.6.2

### 3. ISLR 12.6.2 — Hierarchical Clustering

**(a) Complete Linkage:** Observations 1 and 2 merge at height 0.30, observations 3 and 4 merge at 0.45, and the two clusters merge at 0.80.

**(b) Single Linkage:** Observations 1 and 2 merge at height 0.30, observation 3 joins at 0.40, and observation 4 joins at 0.45.

**(c)** Cutting the complete-linkage dendrogram into two clusters gives {1, 2} and {3, 4}.

**(d)** Cutting the single-linkage dendrogram into two clusters gives {1, 2, 3} and {4}.

**(e)** An equivalent complete-linkage dendrogram can be obtained by swapping the positions of observations 1 and 2. The merge heights and cluster structure remain unchanged.